# 02. CrewAI / AutoGen / ADK

三个框架代表了三种不同的多 Agent 组织方式：

- **CrewAI**：公司部门式 — 角色+任务+流程
- **AutoGen**：研究团队式 — 事件驱动+灵活的 team 组合
- **Google ADK**：企业平台式 — A2A 协议+多语言+Cloud 部署

> **检查点**：能说出三种框架各自最合适的场景（一个场景一句话）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
from dataclasses import dataclass, field
# CrewAI 风格：定义角色和任务，指定流程
@dataclass
class CrewAgent:
    role: str; goal: str; backstory: str; tools: list

@dataclass
class Task:
    description: str; expected_output: str; agent: str
    context: list = None  # 依赖的上游任务

class Crew:
    def __init__(self, agents: list[CrewAgent], tasks: list[Task], process: str = "sequential"):
        self.agents = {a.role: a for a in agents}
        self.tasks = tasks; self.process = process
    
    def kickoff(self):
        results = {}
        for task in self.tasks:
            agent = self.agents[task.agent]
            result = f"[{agent.role}] {task.description} → {task.expected_output}"
            results[task.description] = result
            print(f"  {result}")
        return results

# 定义 crew
crew = Crew(
    agents=[
        CrewAgent("Researcher","Find facts","Expert at gathering data",["search"]),
        CrewAgent("Analyst","Analyze data","Expert at finding patterns",["stats"]),
        CrewAgent("Reporter","Write report","Expert at clear writing",["format"]),
    ],
    tasks=[
        Task("Research market size","Market size in billions","Researcher"),
        Task("Analyze growth rate","CAGR and trends","Analyst"),
        Task("Write executive summary","One-page summary","Reporter"),
    ],
    process="sequential"
)
print("CrewAI-style execution:")
crew.kickoff()

CrewAI-style execution:
  [Researcher] Research market size → Market size in billions
  [Analyst] Analyze growth rate → CAGR and trends
  [Reporter] Write executive summary → One-page summary


{'Research market size': '[Researcher] Research market size → Market size in billions',
 'Analyze growth rate': '[Analyst] Analyze growth rate → CAGR and trends',
 'Write executive summary': '[Reporter] Write executive summary → One-page summary'}

In [3]:
# AutoGen 风格：Team presets + termination conditions
from dataclasses import dataclass, field

@dataclass
class AutoGenAgent:
    name: str; system_message: str
    def generate_reply(self, messages: list) -> str:
        return f"[{self.name}] Response to: {messages[-1][:40]}..."

class Team:
    def __init__(self, agents: list[AutoGenAgent], max_rounds: int = 3):
        self.agents = agents; self.max_rounds = max_rounds
    
    def run(self, task: str):
        messages = [task]
        for r in range(self.max_rounds):
            for agent in self.agents:
                reply = agent.generate_reply(messages)
                messages.append(reply)
                print(f"  Round {r+1} [{agent.name}]: {reply[:60]}...")
            if "TERMINATE" in messages[-1]:
                print("  → Termination condition met")
                break
        return messages

team = Team([
    AutoGenAgent("Analyst","You analyze data objectively"),
    AutoGenAgent("Critic","You find flaws in analysis"),
], max_rounds=2)
print("\nAutoGen-style team:")
team.run("Analyze Q3 sales data")


AutoGen-style team:
  Round 1 [Analyst]: [Analyst] Response to: Analyze Q3 sales data......
  Round 1 [Critic]: [Critic] Response to: [Analyst] Response to: Analyze Q3 sale...
  Round 2 [Analyst]: [Analyst] Response to: [Critic] Response to: [Analyst] Respo...
  Round 2 [Critic]: [Critic] Response to: [Analyst] Response to: [Critic] Respon...


['Analyze Q3 sales data',
 '[Analyst] Response to: Analyze Q3 sales data...',
 '[Critic] Response to: [Analyst] Response to: Analyze Q3 sales ...',
 '[Analyst] Response to: [Critic] Response to: [Analyst] Response...',
 '[Critic] Response to: [Analyst] Response to: [Critic] Response...']

## 三框架决策速查

| | CrewAI | AutoGen | Google ADK |
|---|--------|---------|------------|
| **最佳场景** | 结构化业务流程 | 灵活的研究协作 | 企业级多服务 Agent |
| **部署复杂度** | 中 | 中 | 高（Cloud-native） |
| **跨语言** | Python only | Python + .NET | Python/TS/Go/Java/Kotlin |
| **协议标准** | 内部契约 | 内部消息 | A2A 开放协议 |

> **检查点**：能在一个具体业务需求上，论证为什么选 A 框架而非 B/C。

## 真实 API 实验

CrewAI 风格的 Role+Task 链：用真实 LLM 跑 researcher → writer → reviewer 流水线。


In [4]:
# CrewAI 风格：Role + Task + 顺序执行（真实 LLM）
crew = [
    {"role": "Researcher", "goal": "挖掘事实", "task": "列出 MCP 协议的 2 个核心设计决策"},
    {"role": "Writer", "goal": "清晰表达", "task": "把上面的内容写成 3 句科普"},
    {"role": "Reviewer", "goal": "把关质量", "task": "指出上文最薄弱的一句并改进"},
]
context = ""
for member in crew:
    out = chat(client, f"任务：{member['task']}\n\n已有材料：\n{context or '（无，你是第一棒）'}",
               system=f"你是{member['role']}，目标：{member['goal']}。", max_tokens=200)
    print(f"【{member['role']}】{out}\n")
    context += f"\n[{member['role']} 的产出]\n{out}\n"


【Researcher】根据 MCP 协议的设计文档和公开资料，其核心设计决策包括以下两点：

1. **采用 JSON-RPC 2.0 作为通信协议**  
   为了确保跨语言



【Writer】



【Reviewer】**最薄弱的一句**：  
> “1. **采用 JSON-RPC 2.0 作为通信协议**  
>    为了确保跨语言



## 练习

1. 用 CrewAI 风格实现一个「论文评审」流程：Author 提交 → Reviewer1 + Reviewer2 并行评审 → Editor 综合决定。
2. 在 AutoGen 风格中实现 `SelectorGroupChat`：根据消息内容自动选择下一个发言的 Agent。
3. 调研 A2A 协议的 agent card 格式。设计你自己的 Agent 的 agent card。